### 8. Перейти от одной большой витрины к Снежинке (eBay)

Создать структуру таблиц (с партициями по дате), наполнить измерения уникальными атрибутами, наполнить таблицу фактов.

        DDL-скрипты выполнить через hue/dbeaver.
        DML-скрипты выполнить через Zeppelin или через spark-submit.

Наполнение витрин через Spark-скрипт для миграции данных в Снежинку

Ожидаемый результат:

        таблица фактов
        иерархия категорий (dim_sub_sub_cat, dim_sub_cat, dim_cat)
        таблицы измерений (items, sellers, locations, conditions, currencies, logistics)

---

DDL (выполнения в Hue)

Создадим все измерения и таблицу фактов

---

```sql
use s_yudin;

-- 1. Иерархия категорий
CREATE TABLE IF NOT EXISTS dim_cat (
    cat_id BIGINT,
    category_name STRING
) STORED AS PARQUET;

CREATE TABLE IF NOT EXISTS dim_sub_cat (
    sub_cat_id BIGINT,
    sub_category_name STRING,
    cat_id BIGINT
) STORED AS PARQUET;

CREATE TABLE IF NOT EXISTS dim_sub_sub_cat (
    sub_sub_cat_id BIGINT,
    sub_sub_category_name STRING,
    sub_cat_id BIGINT
) STORED AS PARQUET;

-- 2. Продавцы
CREATE TABLE IF NOT EXISTS dim_sellers (
    seller_id BIGINT,
    seller_name STRING,
    seller_feedback_percentage DOUBLE,
    seller_feedback_score BIGINT,
    top_rated_buying_experience BOOLEAN
) STORED AS PARQUET;

-- 3. Товары (items)
CREATE TABLE IF NOT EXISTS dim_items (
    item_id BIGINT,
    itemid STRING,
    title STRING,
    price DOUBLE,
    currency STRING,
    item_condition STRING,
    buying_options STRING,
    item_creation_ts TIMESTAMP
) STORED AS PARQUET;

-- 4. Локации (только страна)
CREATE TABLE IF NOT EXISTS dim_locations (
    location_id BIGINT,
    location_country STRING
) STORED AS PARQUET;

-- 5. Логистика (условия доставки)
CREATE TABLE IF NOT EXISTS dim_logistics (
    logistics_id BIGINT,
    shipping_cost_type STRING,
    shipping_cost DOUBLE,
    shipping_currency STRING,
    estimated_delivery_days BIGINT
) STORED AS PARQUET;

-- 6. Валюты
CREATE TABLE IF NOT EXISTS dim_currencies (
    currency_id BIGINT,
    currency_code STRING
) STORED AS PARQUET;

-- 7. Таблица фактов (партиционирована по snapshot_dt)
CREATE TABLE IF NOT EXISTS fact_ebay_listings (
    fact_id BIGINT,
    item_id BIGINT,
    seller_id BIGINT,
    location_id BIGINT,
    logistics_id BIGINT,
    currency_id BIGINT,
    sub_sub_cat_id BIGINT,
    price_usd DOUBLE,
    shipping_cost_usd DOUBLE
)
PARTITIONED BY (snapshot_dt DATE)
STORED AS PARQUET;
```

Таблицы отобразились в списке таблиц

![](./../img/ex8.png)

Скопируем файл snowflake_etl.py на кластер.

Выполним spark-submit с следующими параметрами:

```bash
spark-submit --master yarn \
             --deploy-mode cluster \
             --num-executors 3 \
             --executor-memory 2g \
             --conf spark.sql.adaptive.enabled=true \
            snowflake_etl.py
```

К сожалению, соединение было прервано хостом

```bash
...
26/06/10 14:45:21 INFO Client: Application report for application_1781067994033_0034 (state: RUNNING)
26/06/10 14:45:51 INFO Client: Application report for application_1781067994033_0034 (state: RUNNING)
26/06/10 14:46:21 INFO Client: Application report for application_1781067994033_0034 (state: RUNNING)
26/06/10 14:46:52 INFO Client: Application report for application_1781067994033_0034 (state: RUNNING)
Connection to 172.17.1.67 closed by remote host.
Connection to 172.17.1.67 closed.

```

какие-то данные успели внестись в таблицы:

![](./../img/ex8.1.png)


![](./../img/ex8.2.png)

Запустим submit еще раз...


![](./../img/ex8.yarn.png)

соединение прервано - скрипт проработал чуть более 20 минут.

```bash
26/06/10 14:56:12 INFO Client: Application report for application_1781092113564_0003 (state: ACCEPTED)
26/06/10 14:56:15 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:56:15 INFO Client: 
	 client token: N/A
	 diagnostics: N/A
	 ApplicationMaster host: da-hadoop-airflow
	 ApplicationMaster RPC port: 43493
	 queue: root.default
	 start time: 1781092390701
	 final status: UNDEFINED
	 tracking URL: http://da-hadoop-airflow:8088/proxy/application_1781092113564_0003/
	 user: s.yudin
26/06/10 14:56:45 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:57:15 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:57:45 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:58:15 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:58:45 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:59:15 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 14:59:45 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:00:15 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:00:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:01:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:01:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:02:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:02:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:03:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:03:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:04:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:04:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:05:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:05:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:06:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:06:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:07:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:07:46 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:08:16 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:08:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:09:17 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:09:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:10:17 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:10:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:11:17 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:11:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:12:17 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:12:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:13:17 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
26/06/10 15:13:47 INFO Client: Application report for application_1781092113564_0003 (state: RUNNING)
Connection to 172.17.1.67 closed by remote host.
Connection to 172.17.1.67 closed.

```

Загрузим данные поочередно - по одной таблице:

```bash
26/06/10 15:28:02 INFO Client: 
	 client token: N/A
	 diagnostics: N/A
	 ApplicationMaster host: da-hadoop-airflow
	 ApplicationMaster RPC port: 46239
	 queue: root.default
	 start time: 1781094422669
	 final status: SUCCEEDED
	 tracking URL: http://da-hadoop-airflow:8088/proxy/application_1781093807412_0002/
	 user: s.yudin
26/06/10 15:28:02 INFO ShutdownHookManager: Shutdown hook called
26/06/10 15:28:02 INFO ShutdownHookManager: Deleting directory /tmp/spark-de1702e8-b369-4576-baa5-3991b1d05a5b
26/06/10 15:28:02 INFO ShutdownHookManager: Deleting directory /tmp/spark-db2033bf-fac7-40eb-8a67-85036149cede
```

первая таблица загружена.

Некоторе время спустя...

![](./../img/ex8.spark-sub.png)

```bash
s.yudin@da-hadoop-airflow:~$ spark-submit --master yarn \
             --deploy-mode cluster \
             --num-executors 3 \
             --executor-memory 2g \
             --conf spark.sql.adaptive.enabled=true \
            snowflake_etl_9.py 
WARNING: Using incubator modules: jdk.incubator.vector
26/06/10 17:47:49 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/10 17:47:49 INFO DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at /0.0.0.0:8032
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/10 17:47:50 INFO Configuration: resource-types.xml not found
26/06/10 17:47:50 INFO ResourceUtils: Unable to find 'resource-types.xml'.
26/06/10 17:47:50 INFO Client: Verifying our application has not requested more than the maximum memory capability of the cluster (4096 MB per container)
26/06/10 17:47:50 INFO Client: Will allocate AM container, with 1408 MB memory including 384 MB overhead
26/06/10 17:47:50 INFO Client: Setting up container launch context for our AM
26/06/10 17:47:50 INFO Client: Setting up the launch environment for our AM container
26/06/10 17:47:50 INFO Client: Preparing resources for our AM container
26/06/10 17:47:50 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive} is set, falling back to uploading libraries under SPARK_HOME.
26/06/10 17:47:56 INFO Client: Uploading resource file:/tmp/spark-a613f00a-39be-4ce9-9e20-775f9dfab48c/__spark_libs__1444427649514574498.zip -> hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036/__spark_libs__1444427649514574498.zip
26/06/10 17:47:58 INFO Client: Uploading resource file:/home/s.yudin/snowflake_etl_9.py -> hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036/snowflake_etl_9.py
26/06/10 17:47:58 INFO Client: Uploading resource file:/usr/local/spark/python/lib/pyspark.zip -> hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036/pyspark.zip
26/06/10 17:47:58 INFO Client: Uploading resource file:/usr/local/spark/python/lib/py4j-0.10.9.9-src.zip -> hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036/py4j-0.10.9.9-src.zip
26/06/10 17:47:59 INFO Client: Uploading resource file:/tmp/spark-a613f00a-39be-4ce9-9e20-775f9dfab48c/__spark_conf__8063606396157843479.zip -> hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036/__spark_conf__.zip
26/06/10 17:47:59 INFO SecurityManager: Changing view acls to: s.yudin
26/06/10 17:47:59 INFO SecurityManager: Changing modify acls to: s.yudin
26/06/10 17:47:59 INFO SecurityManager: Changing view acls groups to: s.yudin
26/06/10 17:47:59 INFO SecurityManager: Changing modify acls groups to: s.yudin
26/06/10 17:47:59 INFO SecurityManager: SecurityManager: authentication disabled; ui acls disabled; users with view permissions: s.yudin groups with view permissions: EMPTY; users with modify permissions: s.yudin; groups with modify permissions: EMPTY; RPC SSL disabled
26/06/10 17:47:59 INFO Client: Submitting application application_1781093807412_0036 to ResourceManager
26/06/10 17:47:59 INFO YarnClientImpl: Submitted application application_1781093807412_0036
26/06/10 17:48:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:48:00 INFO Client: 
	 client token: N/A
	 diagnostics: [Wed Jun 10 17:47:59 +0300 2026] Application is added to the scheduler and is not yet activated. Skipping AM assignment as cluster resource is empty.  Details : AM Partition = <DEFAULT_PARTITION>; AM Resource Request = <memory:2048, vCores:1>; Queue Resource Limit for AM = <memory:0, vCores:0>; User AM Resource Limit of the queue = <memory:0, vCores:0>; Queue AM Resource Usage = <memory:2048, vCores:1>; 
	 ApplicationMaster host: N/A
	 ApplicationMaster RPC port: -1
	 queue: root.default
	 start time: 1781102879191
	 final status: UNDEFINED
	 tracking URL: http://da-hadoop-airflow:8088/proxy/application_1781093807412_0036/
	 user: s.yudin
26/06/10 17:48:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:49:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:49:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:50:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:50:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:51:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:51:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:52:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:52:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:53:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:53:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:54:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:54:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:55:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:55:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:56:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:56:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:57:00 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:57:30 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:58:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:58:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:59:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 17:59:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:00:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:00:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:01:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:01:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:02:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:02:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:03:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:03:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:04:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:04:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:05:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:05:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:06:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:06:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:07:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:07:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:08:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:08:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:09:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:09:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:10:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:10:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:11:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:11:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:12:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:12:31 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:13:01 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:13:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:14:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:14:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:15:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:15:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:16:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:16:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:17:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:17:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:18:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:18:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:19:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:19:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:20:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:20:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:21:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:21:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:22:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:22:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:23:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:23:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:24:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:24:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:25:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:25:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:26:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:26:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:27:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:27:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:28:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:28:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:29:02 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:29:32 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:30:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:30:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:31:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:31:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:32:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:32:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:33:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:33:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:34:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:34:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:35:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:35:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:36:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:36:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:37:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:37:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:38:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:38:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:39:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:39:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:40:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:40:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:41:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:41:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:42:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:42:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:43:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:43:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:44:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:44:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:45:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:45:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:46:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:46:33 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:47:03 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:47:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:48:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:48:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:49:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:49:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:50:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:50:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:51:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:51:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:52:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:52:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:53:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:53:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:54:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:54:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:55:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:55:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:56:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:56:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:57:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:57:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:58:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:58:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:59:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 18:59:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:00:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:00:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:01:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:01:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:02:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:02:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:03:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:03:34 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:04:04 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:04:35 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:05:05 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:05:35 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:06:05 INFO Client: Application report for application_1781093807412_0036 (state: ACCEPTED)
26/06/10 19:06:24 INFO Client: Application report for application_1781093807412_0036 (state: KILLED)
26/06/10 19:06:24 INFO Client: 
	 client token: N/A
	 diagnostics: Application application_1781093807412_0036 was killed by user dr.who
	 ApplicationMaster host: N/A
	 ApplicationMaster RPC port: -1
	 queue: root.default
	 start time: 1781102879191
	 final status: KILLED
	 tracking URL: http://da-hadoop-airflow:8088/cluster/app/application_1781093807412_0036
	 user: s.yudin
26/06/10 19:06:24 INFO Client: Deleted staging directory hdfs://172.17.1.67:9000/user/s.yudin/.sparkStaging/application_1781093807412_0036
Exception in thread "main" org.apache.spark.SparkException: Application application_1781093807412_0036 is killed
	at org.apache.spark.deploy.yarn.Client.run(Client.scala:1396)
	at org.apache.spark.deploy.yarn.YarnClusterApplication.start(Client.scala:1827)
	at org.apache.spark.deploy.SparkSubmit.org$apache$spark$deploy$SparkSubmit$$runMain(SparkSubmit.scala:1027)
	at org.apache.spark.deploy.SparkSubmit.doRunMain$1(SparkSubmit.scala:204)
	at org.apache.spark.deploy.SparkSubmit.submit(SparkSubmit.scala:227)
	at org.apache.spark.deploy.SparkSubmit.doSubmit(SparkSubmit.scala:96)
	at org.apache.spark.deploy.SparkSubmit$$anon$2.doSubmit(SparkSubmit.scala:1132)
	at org.apache.spark.deploy.SparkSubmit$.main(SparkSubmit.scala:1141)
	at org.apache.spark.deploy.SparkSubmit.main(SparkSubmit.scala)
26/06/10 19:06:24 INFO ShutdownHookManager: Shutdown hook called
26/06/10 19:06:24 INFO ShutdownHookManager: Deleting directory /tmp/spark-a613f00a-39be-4ce9-9e20-775f9dfab48c
26/06/10 19:06:24 INFO ShutdownHookManager: Deleting directory /tmp/spark-f7d968dd-3eef-49f7-ab85-17e0d80416b7

```

```
User: 	s.yudin
Name: 	snowflake_etl_9.py
Application Type: 	SPARK
Application Tags: 	
Application Priority: 	0 (Higher Integer value indicates higher priority)
YarnApplicationState: 	ACCEPTED: waiting for AM container to be allocated, launched and register with RM.
Queue: 	root.default
FinalStatus Reported by AM: 	Application has not completed yet.
Started: 	Wed Jun 10 17:47:59 +0300 2026
Launched: 	N/A
Finished: 	N/A
Elapsed: 	1hrs, 17mins, 42sec
Tracking URL: 	ApplicationMaster
Log Aggregation Status: 	NOT_START
Application Timeout (Remaining Time): 	Unlimited
Diagnostics: 	[Wed Jun 10 18:52:48 +0300 2026] Application is added to the scheduler and is not yet activated. Queue's AM resource limit exceeded. Details : AM Partition = <DEFAULT_PARTITION>; AM Resource Request = <memory:2048, vCores:1>; Queue Resource Limit for AM = <memory:0, vCores:0>; User AM Resource Limit of the queue = <memory:0, vCores:0>; Queue AM Resource Usage = <memory:1024, vCores:1>;
Unmanaged Application: 	false
Application Node Label expression: 	<Not set>
AM container Node Label expression: 	<DEFAULT_PARTITION>
```